[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-lessons/blob/main/docs/ch3/lesson3.ipynb)

# Vegetation & Water Indices
Is it Green? Is it Wet? Calculating and visualizing Normalized Difference Vegetation Index (NDVI) & Normalized Difference Water Index (NDWI) from satellite imagery.

In [1]:
import folium
import ee
import geemap
from datetime import datetime, timedelta

# Authenticate your Google account with Earth Engine
ee.Authenticate()

# Write your project ID here, in quotes
ee.Initialize(project = "emerge-lessons")

In [2]:
def add_ee_layer(self, ee_image_object, vis_params, name):
    """Adds a method for displaying Earth Engine image tiles to folium map."""
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

Draw a circle around an input point, which will be used to clip the satellite image.

In [3]:
# Region of interest
point = ee.Geometry.Point(-81.660044, 28.473813)
region = point.buffer(distance=100000)

Get Sentinel-2 satellite images that match our location and date range, with less than 10% cloud cover.

In [4]:
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(region) \
    .filterDate('2024-04-01', '2024-06-01') \
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 10))

Take the median value of the images in our collection, which helps to remove "messy" data such as clouds.

In [5]:
image = collection.median().clip(region)

Calculate NDVI (normalized difference vegetation index) from the image, which shows how healthy the plants are.
- High NDVI = more green plants
- Formula: (NIR - RED) / (NIR + RED)

In [6]:
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

Calculate NDWI (normalized difference water index) from the image, which shows where the water is.
- High NDWI = more water
- Formula: (GREEN - NIR) / (GREEN + NIR)

In [7]:
ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

Set up an interactive map and center it over Florida.

In [8]:
map = folium.Map(location=[28.473813, -81.660044], tiles="Cartodb dark_matter", zoom_start=8)

Add the NDVI (vegetation) layer to the map, coloring high values green.

In [9]:
map.add_ee_layer(ndvi, {'min': 0, 'max': 1, 'palette': ['white', 'green']}, 'NDVI')

Add the NDWI (water) layer to the map, coloring high values blue.

In [10]:
map.add_ee_layer(ndwi, {'min': -1, 'max': 1, 'palette': ['white', 'blue']}, 'NDWI')

Show the map, with a layer control section in the top right to allow us to show or hide each layer.

In [ ]:
folium.LayerControl(collapsed = False).add_to(map)
display(map)

In the interactive map above, click on the check box next to `NDWI` to hide the layer, showing `NDVI` below it. You can click the check box again to show `NDWI` again.